# Week 6, Lab 3 — Local-model agent as MCP client


In [ ]:
WEEK = 'Week 6'
LAB = 'Lab 3 — agent as MCP client'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "local_tools_server.py")])

async def mcp_tools():
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            listed = await s.list_tools()
            return [{"name": t.name, "description": t.description} for t in listed.tools]

schemas = await mcp_tools()
print(schemas)

SYSTEM = "You are a tool-using agent. Tools:\n" + "\n".join(f"- {t['name']}: {t['description']}" for t in schemas)
SYSTEM += '\nIf you need a tool, reply ONLY JSON {"name": "...", "arguments": {...}}. Else answer the user.'

async def run(question: str) -> str:
    reply = local_chat([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ], max_new_tokens=120, temperature=0.1)
    print("MODEL", reply)
    call = parse_tool_call(reply)
    if not call:
        return reply
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            result = await s.call_tool(call["name"], call["arguments"])
    print("MCP", result)
    return local_chat([
        {"role": "system", "content": "Answer using the tool result. Brief."},
        {"role": "user", "content": question},
        {"role": "user", "content": f"TOOL RESULT: {result}"},
    ], max_new_tokens=120, temperature=0.2)

print(await run("What is 11*13?"))
print(await run("What is MCP?"))


Same LLM loop as Week 1; only the tool transport changed.
